# Whole-dataset figures with hu-neuro-pipeline

This notebook calls the installed **hu-neuro-pipeline `group_pipeline()`** to process all
13 participants and their 3 runs. The library performs filtering, average referencing,
epoching, baseline correction, amplitude rejection, participant averages and grand averages.
The remaining cells prepare this dataset's input format and recreate the original figures
from the library outputs. There is no separate custom processing module.

Run all cells in order. Set `RUN_PROCESSING=False` after the first successful run to reuse
the library results for plotting. Set it back to `True` whenever settings or inputs change.
The original single-recording notebook remains in `notebook_backups/figure replications.single-run.ipynb`.

Library: [quickstart](https://hu-neuro-pipeline.readthedocs.io/en/stable/quickstart_py.html)
and [settings](https://hu-neuro-pipeline.readthedocs.io/en/stable/inputs_py.html).

In [ ]:
# Optional, once in this notebook's kernel environment:
# %pip install hu-neuro-pipeline==0.9.0 matplotlib

from pathlib import Path
from importlib.metadata import version
import json
import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pipeline import group_pipeline

PROJECT_DIR = Path.cwd().resolve()  # Set this if the kernel starts elsewhere.
DATA_DIR = PROJECT_DIR / 'ds003061'
OUTPUT_DIR = PROJECT_DIR / 'hu_neuro_results'
INPUT_DIR = OUTPUT_DIR / 'prepared_inputs'
FIGURE_DIR = OUTPUT_DIR / 'figures'
for folder in (OUTPUT_DIR, INPUT_DIR, FIGURE_DIR):
    folder.mkdir(parents=True, exist_ok=True)

print('hu-neuro-pipeline:', version('hu-neuro-pipeline'), '| MNE:', mne.__version__)

In [ ]:
RUN_PROCESSING = True
SUBJECTS = None  # None = everyone; or e.g. ['sub-001', 'sub-002']
CONDITIONS = ['standard', 'oddball', 'noise']
CHANNEL = 'Pz'
PEAK_WINDOW = (0.250, 0.500)
COLORS = dict(standard='tab:blue', oddball='tab:orange', noise='tab:green')

# Keep the original notebook's reference, filtering and epoch/baseline windows.
# 200 µV is hu-neuro-pipeline's default rejection threshold; None disables it.
SETTINGS = dict(
    montage='biosemi64', ref_channels='average',
    veog_channels=None, heog_channels=None,
    highpass_freq=1.0, lowpass_freq=None,
    epochs_tmin=-0.2, epochs_tmax=0.8, baseline=(-0.2, 0.0),
    reject_peak_to_peak=200.0,
    bad_channels=None, ica_method=None,
    triggers=[1, 2, 3], triggers_column='trigger',
    components=dict(name=['P300'], tmin=[0.250], tmax=[0.500], roi=[['Pz']]),
    average_by={condition: f'condition == "{condition}"' for condition in CONDITIONS},
    to_df=False, n_jobs=1,
)

## Prepare the dataset format

The library expects numeric trigger annotations and a matching trial log. The dataset uses
named sounds, so the following cell converts standard/oddball/noise to 1/2/3 and creates
matching CSV logs. Response-tagged sounds stay with their sound type, including the spelling
`reponse`; separate button responses and glitch markers are excluded. Only the 64 scalp
channels are copied to the prepared FIF inputs. Source `.set` files are unchanged.

This cell applies **no filtering, referencing, ICA, rejection threshold or averaging**.
Sounds whose analysis window extends outside their recording are omitted from both annotations
and logs before passing the runs to the library. Each participant's three runs are supplied as
a nested list, so hu-neuro-pipeline pools their trials before computing its equal-participant
grand average. Prepared inputs and saved epochs require several GB of disk space.

In [ ]:
participants = pd.read_csv(DATA_DIR / 'participants.tsv', sep='	').participant_id.tolist()
subjects = sorted(participants if SUBJECTS is None else SUBJECTS)
assert subjects and len(subjects) == len(set(subjects)) and set(subjects) <= set(participants)
scalp_channels = mne.channels.make_standard_montage('biosemi64').ch_names
codes = {label: code for code, condition in enumerate(CONDITIONS, 1)
         for label in (condition, condition + '_with_reponse', condition + '_with_response')}
ignored = {'response', 'ignore', 'condition 5', 'condition_5'}
raw_files, log_files, inventory, library_ids = [], [], [], {}

for subject in subjects:
    runs = sorted((DATA_DIR / subject / 'eeg').glob(f'{subject}_task-P300_run-*_eeg.set'))
    assert [int(p.name.split('_run-')[1].split('_')[0]) for p in runs] == [1, 2, 3], subject
    prepared_runs, trial_logs = [], []
    for run, source in enumerate(runs, 1):
        table = pd.read_csv(str(source).replace('_eeg.set', '_events.tsv'), sep='	')
        assert not (set(table.value) - set(codes) - ignored), source
        sounds = table.loc[table.value.isin(codes)].copy()
        # Inspect without loading the signal unless the format conversion is needed.
        raw = mne.io.read_raw_eeglab(source, preload=False, verbose='ERROR')
        samples = np.rint(sounds.onset.to_numpy() * raw.info['sfreq']).astype(int)
        valid = ((samples + round(SETTINGS['epochs_tmin'] * raw.info['sfreq']) >= 0) &
                 (samples + round(SETTINGS['epochs_tmax'] * raw.info['sfreq']) < raw.n_times))
        sounds = sounds.loc[valid].copy()
        sounds['trigger'] = sounds.value.map(codes).astype(int)
        sounds['condition'] = sounds.trigger.map(dict(enumerate(CONDITIONS, 1)))
        sounds['subject'] = subject
        sounds['run'] = run
        prepared = INPUT_DIR / f'{subject}_run-{run}_raw.fif'
        stamp_file = prepared.with_suffix('.json')
        stamp = dict(source=str(source), bytes=source.stat().st_size,
                     mtime_ns=source.stat().st_mtime_ns,
                     events_mtime_ns=Path(str(source).replace('_eeg.set', '_events.tsv')).stat().st_mtime_ns,
                     tmin=SETTINGS['epochs_tmin'], tmax=SETTINGS['epochs_tmax'], channels=scalp_channels)
        if not (prepared.exists() and stamp_file.exists() and json.loads(stamp_file.read_text()) == stamp):
            raw.pick(scalp_channels).load_data()
            raw.set_annotations(mne.Annotations(sounds.onset, np.zeros(len(sounds)), sounds.trigger.astype(str)))
            raw.save(prepared, overwrite=True, verbose='ERROR')
            stamp_file.write_text(json.dumps(stamp, indent=2), encoding='utf-8')
        raw.close()
        prepared_runs.append(str(prepared))
        trial_logs.append(sounds[['subject', 'run', 'condition', 'trigger', 'value', 'onset']])
        inventory.append(dict(subject=subject, run=run, source=str(source),
                              n_sounds=int(valid.size), n_outside_window=int((~valid).sum()),
                              n_passed_to_library=len(sounds)))
    log_path = INPUT_DIR / f'{subject}_trials.csv'
    pd.concat(trial_logs, ignore_index=True).to_csv(log_path, index=False)
    raw_files.append(prepared_runs)
    log_files.append(str(log_path))
    library_ids['_'.join(Path(p).stem for p in prepared_runs)] = subject
    print(f'Prepared {subject}: {sum(len(t) for t in trial_logs)} sound trials', flush=True)

inventory = pd.DataFrame(inventory)
inventory.to_csv(OUTPUT_DIR / 'input_inventory.csv', index=False)
print(f'{len(subjects)} participants; {len(inventory)} recordings')
display(inventory)

## Run hu-neuro-pipeline

This is the library's processing entry point. `trials.csv`, `ave.csv`, `grand_ave.fif`,
 `config.json`, and the saved epochs are produced by hu-neuro-pipeline.
`P300` in the trial table is its single-trial mean amplitude at Pz over 250–500 ms;
rejected trials have a missing P300 value. ICA is disabled explicitly: the manually selected
components from participant 001 are not transferred to other recordings.

hu-neuro-pipeline removes the epoch's final sample; its epoch output therefore has 256
time points here, rather than the original notebook's 257. Inspect the retained-trial
counts below before interpreting results.

In [ ]:
if RUN_PROCESSING:
    trials, evokeds_df, pipeline_config = group_pipeline(
        raw_files=raw_files, log_files=log_files,
        output_dir=str(OUTPUT_DIR), epochs_dir=str(OUTPUT_DIR / 'epochs'),
        **SETTINGS,
    )
else:
    assert (OUTPUT_DIR / 'config.json').exists(), 'Set RUN_PROCESSING=True for the first run.'
    trials = pd.read_csv(OUTPUT_DIR / 'trials.csv')
    evokeds_df = pd.read_csv(OUTPUT_DIR / 'ave.csv')
    pipeline_config = json.loads((OUTPUT_DIR / 'config.json').read_text())

assert set(trials.participant_id.unique()) == set(library_ids), 'Results do not match selected subjects.'
assert set(evokeds_df.participant_id.unique()) == set(library_ids)
assert evokeds_df.groupby('participant_id').label.nunique().eq(3).all(), 'A condition is missing.'
grand_evokeds = {e.comment: e for e in mne.read_evokeds(OUTPUT_DIR / 'grand_ave.fif', verbose='ERROR')}
assert set(grand_evokeds) == set(CONDITIONS)
print('Library processing complete:', len(library_ids), 'participants')

In [ ]:
trials['retained'] = trials.P300.notna()
qc = trials.groupby(['subject', 'run', 'condition']).agg(
    n_epoched=('retained', 'size'), n_retained=('retained', 'sum')).reset_index()
qc['retained_percent'] = 100 * qc.n_retained / qc.n_epoched
qc.to_csv(OUTPUT_DIR / 'trial_retention.csv', index=False)
display(qc.groupby('condition')[['n_epoched', 'n_retained']].sum().reindex(CONDITIONS))
flagged = qc.loc[(qc.n_retained < 20) | (qc.retained_percent < 50)]
print('Run/condition rows flagged for review:', len(flagged))
display(flagged)

retention = qc.groupby(['subject', 'condition'])[['n_epoched', 'n_retained']].sum()
retention = (100 * retention.n_retained / retention.n_epoched).unstack().reindex(columns=CONDITIONS)
ax = retention.plot.bar(figsize=(11, 4), color=[COLORS[c] for c in CONDITIONS])
ax.set(ylabel='Retained trials (%)', ylim=(0, 105), xlabel='Participant')
ax.legend(loc='upper left', bbox_to_anchor=(1, 1))
fig = ax.figure
fig.tight_layout()
for extension in ('png', 'pdf'):
    fig.savefig(FIGURE_DIR / f'trial_retention.{extension}', dpi=180, bbox_inches='tight')
plt.show()

## Grand-average ERP figures

Plot hu-neuro-pipeline's grand-average FIF, with ±1 SEM across its participant averages.
Voltage columns in the library's `evokeds_df` are already in µV; its `time` column is in seconds.
The separate grand-average MNE objects store volts. No trials are pooled across participants.

In [ ]:
for figure_name, conditions, xlim in [
    ('erp_standard_oddball', ['standard', 'oddball'], None),
    ('erp_standard_oddball_0_800ms', ['standard', 'oddball'], (0, 800)),
    ('erp_all_conditions', CONDITIONS, None),
]:
    fig, ax = plt.subplots(figsize=(9, 4.5))
    for condition in conditions:
        evoked = grand_evokeds[condition]
        voltage = evoked.copy().pick(CHANNEL).data[0] * 1e6
        participant_values = evokeds_df.loc[evokeds_df.label.eq(condition)]
        errors = participant_values.groupby('time')[CHANNEL].sem().to_numpy()
        assert len(errors) == len(voltage)
        ax.plot(evoked.times * 1000, voltage, color=COLORS[condition], label=condition.capitalize())
        if len(subjects) > 1:
            ax.fill_between(evoked.times * 1000, voltage - errors, voltage + errors,
                            color=COLORS[condition], alpha=0.15)
    ax.axvspan(250, 500, color='grey', alpha=0.12)
    ax.axhline(0, color='black', linewidth=0.5)
    ax.axvline(0, color='black', linewidth=0.5)
    ax.set(xlabel='Time after sound (ms)', ylabel='Amplitude (µV)', xlim=xlim,
           title=f'hu-neuro-pipeline grand average at {CHANNEL} | N={len(subjects)}; shading=±1 SEM')
    ax.legend()
    fig.tight_layout()
    for extension in ('png', 'pdf'):
        fig.savefig(FIGURE_DIR / f'{figure_name}.{extension}', dpi=180, bbox_inches='tight')
    plt.show()

## Peak amplitude and latency

Use each participant waveform produced by the library to find its maximum signed voltage
in 250–500 ms. The bars show means of participant peaks with ±1 SEM and individual dots.
These differ from finding the maximum of the grand-average waveform. Negative maxima are
retained if a participant's waveform never becomes positive in the window.

In [ ]:
window = evokeds_df.loc[evokeds_df.time.between(*PEAK_WINDOW)]
indices = window.groupby(['participant_id', 'label'])[CHANNEL].idxmax()
peak_results = window.loc[indices, ['participant_id', 'label', 'time', CHANNEL]].copy()
peak_results['subject'] = peak_results.participant_id.map(library_ids)
peak_results = peak_results.rename(columns={'label': 'condition', CHANNEL: 'amplitude_uv'})
peak_results['latency_ms'] = peak_results.pop('time') * 1000
peak_results.to_csv(OUTPUT_DIR / 'participant_peaks.csv', index=False)
display(peak_results)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, column, ylabel in zip(axes, ['amplitude_uv', 'latency_ms'], ['Peak amplitude (µV)', 'Peak latency (ms)']):
    for index, condition in enumerate(CONDITIONS):
        values = peak_results.loc[peak_results.condition.eq(condition), column]
        ax.bar(index, values.mean(), color=COLORS[condition], alpha=.65)
        if len(values) > 1:
            ax.errorbar(index, values.mean(), yerr=values.sem(), color='black', capsize=4)
        ax.scatter(index + np.linspace(-.09, .09, len(values)), values, color='black', s=12)
    ax.set(xticks=range(3), xticklabels=[c.capitalize() for c in CONDITIONS], ylabel=ylabel)
fig.suptitle(f'{CHANNEL}, 250–500 ms | participant peaks, mean ± SEM')
fig.tight_layout()
for extension in ('png', 'pdf'):
    fig.savefig(FIGURE_DIR / f'peak_comparison.{extension}', dpi=180, bbox_inches='tight')
plt.show()

In [ ]:
grand_peak_rows = []
for condition in CONDITIONS:
    erp = grand_evokeds[condition].copy().pick(CHANNEL)
    mask = (erp.times >= PEAK_WINDOW[0]) & (erp.times <= PEAK_WINDOW[1])
    voltage = erp.data[0, mask] * 1e6
    peak = int(np.argmax(voltage))
    grand_peak_rows.append(dict(condition=condition, amplitude_uv=voltage[peak],
                                latency_ms=erp.times[mask][peak] * 1000))
grand_peaks = pd.DataFrame(grand_peak_rows)
grand_peaks.to_csv(OUTPUT_DIR / 'grand_average_peaks.csv', index=False)
display(grand_peaks)

## Scalp maps

Use the library's grand-average waveforms. Each set uses the same symmetric color limits
across conditions and requested times; MNE plots the nearest available sample.

In [ ]:
for name, times in [('early', [0.020, 0.050]), ('late', [0.100, 0.200, 0.350, 0.500])]:
    limit = max(np.abs(erp.copy().pick('eeg').data[:, [np.abs(erp.times-t).argmin() for t in times]]).max()
                for erp in grand_evokeds.values()) * 1e6
    for condition in CONDITIONS:
        fig = grand_evokeds[condition].plot_topomap(
            times=times, ch_type='eeg', time_unit='ms', cmap='RdBu_r',
            vlim=(-limit, limit), colorbar=True, show=False)
        fig.suptitle(f'{condition.capitalize()} | hu-neuro-pipeline grand average, N={len(subjects)}')
        for extension in ('png', 'pdf'):
            fig.savefig(FIGURE_DIR / f'topomap_{name}_{condition}.{extension}', dpi=180, bbox_inches='tight')
    plt.show()

## Mean EEG power

Compute Welch PSD with MNE from the epochs saved by hu-neuro-pipeline, selecting the
same retained trials used for its averages (`P300` not missing). Use all 64 scalp electrodes,
average trials/electrodes within participants, then average participants in linear power
before converting to dB relative to 1 µV²/Hz. This is total epoch power, not the spectrum of
the averaged ERP and not baseline-normalized time-frequency power.

In [ ]:
participant_power = {condition: [] for condition in CONDITIONS}
for library_id, subject in library_ids.items():
    epochs = mne.read_epochs(OUTPUT_DIR / 'epochs' / f'{library_id}_epo.fif', preload=False, verbose='ERROR')
    for condition in CONDITIONS:
        good = epochs.metadata.P300.notna() & epochs.metadata.condition.eq(condition)
        assert good.any(), f'{subject}: no retained {condition} trials'
        spectrum = epochs[np.flatnonzero(good)].compute_psd(
            method='welch', fmin=1, fmax=40, n_fft=256, n_overlap=128, picks='eeg', verbose='ERROR')
        power, frequencies = spectrum.get_data(return_freqs=True)
        participant_power[condition].append(power.mean(axis=(0, 1)))
    del epochs

fig, ax = plt.subplots(figsize=(9, 4))
psd_rows = []
for condition in CONDITIONS:
    power_uv2 = np.mean(participant_power[condition], axis=0) * 1e12
    power_db = 10 * np.log10(power_uv2)
    ax.plot(frequencies, power_db, color=COLORS[condition], label=condition.capitalize())
    psd_rows.append(pd.DataFrame(dict(condition=condition, frequency_hz=frequencies,
                                      power_uv2_hz=power_uv2, power_db=power_db)))
pd.concat(psd_rows, ignore_index=True).to_csv(OUTPUT_DIR / 'mean_eeg_power.csv', index=False)
ax.set(xlabel='Frequency (Hz)', ylabel='PSD (dB re 1 µV²/Hz)', title=f'Mean scalp EEG power | N={len(subjects)}')
ax.legend()
fig.tight_layout()
for extension in ('png', 'pdf'):
    fig.savefig(FIGURE_DIR / f'mean_eeg_power.{extension}', dpi=180, bbox_inches='tight')
plt.show()

## Saved outputs

`hu_neuro_results/config.json` contains the library settings, input files, software versions
and its rejection decisions. `input_inventory.csv` records the format conversion and sound
windows that fall outside the recording. The library's `trials.csv`, `ave.csv`, 
`grand_ave.fif` and `epochs/` hold the analysis data. `figures/` contains PNG/PDF figure pairs;
the extra peak and power CSVs contain the values used in those figures.

These task events label sound conditions, not meditation states. The default run does not
apply the single-recording notebook's manually selected ICA components, and does not claim
to reproduce a published preprocessing protocol. Review the library's QC and settings for
your analysis.

In [ ]:
print('Results:', OUTPUT_DIR)
print('Figures:', FIGURE_DIR)
print('PNG/PDF figure pairs:', len(list(FIGURE_DIR.glob('*.png'))))

# Paper-layout comparisons and numerical differences

These cells reuse **hu-neuro-pipeline outputs**, keeping the processing above intact.
Run this section independently to redraw paper-style panels. Original images are attributed
references; recomputed panels retain their measured values. Matching a layout does not
establish numerical reproduction.

[Attar (2025), DOI: 10.3389/fnhum.2025.1616456](https://pmc.ncbi.nlm.nih.gov/articles/PMC12417730/)
is distributed under CC BY 4.0. Original images and source URLs are in `paper_references/`.

In [ ]:
from pathlib import Path
import hashlib
import json
import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

PROJECT_DIR = Path.cwd().resolve()  # Use the same project folder as above.
RESULTS = PROJECT_DIR / 'hu_neuro_results'
REFERENCES = PROJECT_DIR / 'paper_references'
PAPER_DIR = RESULTS / 'paper_comparison'
PAPER_DIR.mkdir(exist_ok=True)
PAPER_CONDITIONS = ['standard', 'oddball']
PAPER_CHANNEL = 'Pz'  # Explicit choice: the article does not identify its ERP plotting channel.
PAPER_WINDOW = (0.250, 0.500)
paper_grands = {e.comment: e for e in mne.read_evokeds(RESULTS / 'grand_ave.fif', verbose='ERROR')}
paper_evokeds = pd.read_csv(RESULTS / 'ave.csv')
paper_peaks = pd.read_csv(RESULTS / 'participant_peaks.csv')
paper_psd = pd.read_csv(RESULTS / 'mean_eeg_power.csv')
paper_subjects = sorted(paper_peaks.subject.unique())
paper_colors = dict(standard='orange', oddball='blue')

def save_paper_figure(fig, name):
    for extension in ('png', 'pdf'):
        fig.savefig(PAPER_DIR / f'{name}.{extension}', dpi=180, bbox_inches='tight')
    return fig

def paper_grid(ax):
    ax.grid(True, linestyle=':', linewidth=.6, color='.85')
    ax.spines[['top', 'right']].set_visible(False)
    ax.tick_params(direction='in')

def compare_reference(number, recomputed_names, labels):
    files = [REFERENCES / f'figure_{number:02d}.jpg'] + [PAPER_DIR / f'{name}.png' for name in recomputed_names]
    height = 7 * max(Image.open(filename).height / Image.open(filename).width for filename in files) + .55
    fig, axes = plt.subplots(1, 1 + len(recomputed_names), figsize=(7 * (1 + len(recomputed_names)), height))
    for ax, filename, label in zip(axes, files, ['Published reference'] + labels):
        ax.imshow(plt.imread(filename))
        ax.set_title(label, fontsize=12)
        ax.axis('off')
    fig.text(.5, .015, 'Original: Attar (2025), doi:10.3389/fnhum.2025.1616456, CC BY 4.0. Recomputed panels: local ds003061 data.',
             ha='center', fontsize=9)
    fig.tight_layout(rect=(0, .04, 1, 1))
    return save_paper_figure(fig, f'comparison_figure{number:02d}')

print('Loaded saved library results for', len(paper_subjects), 'participants; no preprocessing rerun.')

## Figure 2: amplitude and latency panels

The published panel has five positions without a participant mapping. The first plot below
uses the first five local subjects solely to match its five-position layout; the CSV records
that mapping. An additional panel includes everyone. Bars use the existing participant Pz
peaks at 250–500 ms. No published bar values are substituted into the analysis.

In [ ]:
for name, selected in [('figure02_five_positions', paper_subjects[:5]), ('figure02_all_participants', paper_subjects)]:
    mapping = pd.DataFrame(dict(position=[f'P{i+1}' for i in range(len(selected))], subject=selected))
    mapping.to_csv(PAPER_DIR / f'{name}_mapping.csv', index=False)
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.77))
    positions = np.arange(len(selected))
    for ax, column, ylabel, title in zip(axes, ['amplitude_uv', 'latency_ms'],
                                       ['Amplitude (µV)', 'Latency (ms)'],
                                       ['Amplitude Comparison', 'Latency Comparison']):
        values = paper_peaks.pivot(index='subject', columns='condition', values=column).loc[selected]
        odd = ax.bar(positions, values.oddball, color='gold', width=.8, label='Oddball ' + title.split()[0])
        standard = ax.bar(positions, values.standard, color='orange', alpha=.85, width=.8,
                          label='Standard ' + title.split()[0])
        ax.set(xticks=positions, xticklabels=mapping.position, ylabel=ylabel, title=title)
        ax.legend(handles=[standard, odd], fontsize=8, loc='upper right' if column=='amplitude_uv' else 'lower left')
        paper_grid(ax)
    fig.text(.5, .005, f'Recomputed {PAPER_CHANNEL} peaks, 250–500 ms; position-to-subject mapping saved alongside this figure.', ha='center', fontsize=8)
    fig.tight_layout(rect=(0, .03, 1, 1))
    save_paper_figure(fig, name)
    plt.show()
compare_reference(2, ['figure02_five_positions'], ['Recomputed: first five local participants'])
plt.show()

## Figure 3: waveform layout and image-derived comparison

Blue/orange lines and cross markers follow the published layout. Here crosses mark each
waveform's maximum over the displayed 0–800 ms, separately from the P300-window measurements
above. Reference traces below are approximate pixel digitizations, not the authors' numerical
arrays. Calibration and image resolution are saved so the estimates can be audited.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.32))
observed_waveforms = []
for condition in ['oddball', 'standard']:
    evoked = paper_grands[condition].copy().pick(PAPER_CHANNEL)
    mask = (evoked.times >= 0) & (evoked.times <= .8)
    time_ms = evoked.times[mask] * 1000
    voltage = evoked.data[0, mask] * 1e6
    color = paper_colors[condition]
    ax.plot(time_ms, voltage, color=color, linewidth=1.2, label=condition.capitalize() + ' Session ERP')
    peak = np.argmax(voltage)
    ax.plot(time_ms[peak], voltage[peak], 'x', color=color, markersize=8,
            label=condition.capitalize() + ' Peaks')
    observed_waveforms.append(pd.DataFrame(dict(condition=condition, time_ms=time_ms, amplitude_uv=voltage)))
ax.set(xlim=(0, 800), xlabel='Time (ms)', ylabel='Amplitude (µV)')
handles, labels = ax.get_legend_handles_labels()
order = [0,2,1,3]
ax.legend([handles[i] for i in order], [labels[i] for i in order], loc='upper right', fontsize=9)
paper_grid(ax)
fig.tight_layout()
save_paper_figure(fig, 'figure03_erp')
plt.show()
observed_waveforms = pd.concat(observed_waveforms, ignore_index=True)
observed_waveforms.to_csv(PAPER_DIR / 'recomputed_figure03_waveforms.csv', index=False)
compare_reference(3, ['figure03_erp'], ['Recomputed: Pz, all participants'])
plt.show()

In [ ]:
# Pixel anchors refer to the unchanged publisher JPGs downloaded with this notebook.
calibrations = {
    'figure03': dict(x_zero=77., x_end=677., x_value_end=800., y_reference=205.,
                     y_value_reference=0., pixels_per_y_unit=57., x_min=77, x_max=677,
                     y_min=8, y_max=303, legend_x=510, legend_y=100,
                     x_units='ms', y_units='µV', axis_anchor_uncertainty_pixels=2),
    'figure06': dict(x_zero=60., x_end=717., x_value_end=30., y_reference=299.,
                     y_value_reference=40., pixels_per_y_unit=38.5, x_min=80, x_max=718,
                     y_min=12, y_max=415, legend_x=570, legend_y=70,
                     x_units='Hz', y_units='published log-power scale; density normalization unspecified',
                     axis_anchor_uncertainty_pixels=2),
}
calibrations['method_note'] = 'Column medians summarize all matching curve-colored pixels. Column pixel envelopes are saved separately; line thickness, vertical excursions and JPEG artifacts add uncertainty beyond axis-anchor placement. No author arrays are inferred.'

def digitize_reference(number, calibration):
    rgb = np.array(Image.open(REFERENCES / f'figure_{number:02d}.jpg').convert('RGB'))
    red, green, blue = [rgb[..., i].astype(float) for i in range(3)]
    masks = dict(oddball=(blue > 140) & (red < 100) & (green < 120),
                 standard=(red > 170) & (green > 85) & (green < 230) & (blue < 110))
    traces = []
    for condition, mask in masks.items():
        mask[:calibration['y_min']] = False
        mask[calibration['y_max']+1:] = False
        mask[:calibration['legend_y'], calibration['legend_x']:] = False
        xs, ys, top, bottom = [], [], [], []
        for x in range(calibration['x_min'], calibration['x_max']+1):
            rows = np.flatnonzero(mask[:, x])
            if len(rows):
                xs.append(x)
                ys.append(np.median(rows))
                top.append(rows.min())
                bottom.append(rows.max())
        assert len(xs) > 400, f'Insufficient digitized points: Figure {number}, {condition}'
        x = (np.array(xs)-calibration['x_zero']) / (calibration['x_end']-calibration['x_zero']) * calibration['x_value_end']
        y = calibration['y_value_reference'] + (calibration['y_reference']-np.array(ys)) / calibration['pixels_per_y_unit']
        traces.append(pd.DataFrame(dict(condition=condition, x=x, published_plot_value_approx=y,
            published_column_low=calibration['y_value_reference']+(calibration['y_reference']-np.array(bottom))/calibration['pixels_per_y_unit'],
            published_column_high=calibration['y_value_reference']+(calibration['y_reference']-np.array(top))/calibration['pixels_per_y_unit'],
            pixel_x=xs, pixel_y=ys)))
    return pd.concat(traces, ignore_index=True)

reference_erp = digitize_reference(3, calibrations['figure03'])
reference_psd = digitize_reference(6, calibrations['figure06'])
reference_erp.to_csv(PAPER_DIR / 'digitized_paper_figure03.csv', index=False)
reference_psd.to_csv(PAPER_DIR / 'digitized_paper_figure06.csv', index=False)
(PAPER_DIR / 'digitization_calibration.json').write_text(json.dumps(calibrations, indent=2), encoding='utf-8')

peak_differences = []
for condition in PAPER_CONDITIONS:
    ref = reference_erp.loc[reference_erp.condition.eq(condition)]
    obs = observed_waveforms.loc[observed_waveforms.condition.eq(condition)]
    ref_peak = ref.loc[ref.published_plot_value_approx.idxmax()]
    obs_peak = obs.loc[obs.amplitude_uv.idxmax()]
    peak_differences.append(dict(condition=condition,
        published_peak_uv_approx=ref_peak.published_plot_value_approx,
        published_peak_ms_approx=ref_peak.x,
        published_peak_column_envelope_low_uv=ref.published_column_low.max(),
        published_peak_column_envelope_high_uv=ref.published_column_high.max(),
        recomputed_peak_uv=obs_peak.amplitude_uv, recomputed_peak_ms=obs_peak.time_ms,
        amplitude_difference_uv=obs_peak.amplitude_uv-ref_peak.published_plot_value_approx,
        latency_difference_ms=obs_peak.time_ms-ref_peak.x))
peak_differences = pd.DataFrame(peak_differences)
peak_differences.to_csv(PAPER_DIR / 'figure03_peak_differences.csv', index=False)
display(peak_differences)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, condition in zip(axes, PAPER_CONDITIONS):
    reference = reference_erp.loc[reference_erp.condition.eq(condition)]
    observed = observed_waveforms.loc[observed_waveforms.condition.eq(condition)]
    ax.plot(reference.x, reference.published_plot_value_approx, color='.4', linestyle='--', label='Published image: approximate trace')
    ax.plot(observed.time_ms, observed.amplitude_uv, color=paper_colors[condition], label='Recomputed Pz')
    ax.set(xlim=(0,800), xlabel='Time (ms)', ylabel='Amplitude (µV)', title=condition.capitalize())
    paper_grid(ax)
    ax.legend(fontsize=8)
fig.tight_layout()
save_paper_figure(fig, 'figure03_trace_differences')
plt.show()

## Figure 6: spectrum layout

Use the already saved condition PSDs, limiting the display to 1–30 Hz and retaining the
blue/orange styling. The article's axis specifies log power without a per-Hz denominator;
ours is explicitly power spectral density. The overlay compares displayed scales without
claiming those unit definitions are interchangeable or adding an offset to align them.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.51))
for condition in ['oddball', 'standard']:
    values = paper_psd.loc[paper_psd.condition.eq(condition) & paper_psd.frequency_hz.between(1,30)]
    ax.plot(values.frequency_hz, values.power_db, color=paper_colors[condition], linewidth=1.3,
            label=condition.capitalize() + ' Session')
ax.set(xlim=(0,30), xlabel='Frequency (Hz)', ylabel='Log power (dB re 1 µV²/Hz)')
ax.legend(loc='upper right')
paper_grid(ax)
fig.tight_layout()
save_paper_figure(fig, 'figure06_psd')
plt.show()
compare_reference(6, ['figure06_psd'], ['Recomputed: measured PSD scale'])
plt.show()

fig, axes = plt.subplots(1,2,figsize=(11,4))
for ax, condition in zip(axes, PAPER_CONDITIONS):
    ref = reference_psd.loc[reference_psd.condition.eq(condition)]
    observed = paper_psd.loc[paper_psd.condition.eq(condition) & paper_psd.frequency_hz.between(1,30)]
    ax.plot(ref.x, ref.published_plot_value_approx, '--', color='.4', label='Published plot scale (approximate)')
    ax.plot(observed.frequency_hz, observed.power_db, color=paper_colors[condition], label='Recomputed dB re 1 µV²/Hz')
    ax.set(xlim=(0,30), xlabel='Frequency (Hz)', ylabel='Displayed log-power value', title=condition.capitalize())
    paper_grid(ax)
    ax.legend(fontsize=8)
fig.suptitle('Scale comparison: published power normalization is unspecified')
fig.tight_layout()
save_paper_figure(fig, 'figure06_scale_difference')
plt.show()

scale_rows = []
for condition in PAPER_CONDITIONS:
    ref = reference_psd.loc[reference_psd.condition.eq(condition)].sort_values('x')
    observed = paper_psd.loc[paper_psd.condition.eq(condition) & paper_psd.frequency_hz.between(1,30)]
    reference_values = np.interp(observed.frequency_hz, ref.x, ref.published_plot_value_approx)
    scale_rows.append(pd.DataFrame(dict(condition=condition, frequency_hz=observed.frequency_hz,
        published_plot_value_approx=reference_values, recomputed_db_uv2_per_hz=observed.power_db,
        numeric_difference_not_unit_harmonized=observed.power_db-reference_values)))
pd.concat(scale_rows,ignore_index=True).to_csv(PAPER_DIR/'figure06_display_scale_comparison.csv',index=False)

## Figures 4–5: both interpretations of the maps

Their captions name 20/50 ms ERPs, but the image titles name 4–30 Hz spectra. The following
plots show both interpretations for standard and oddball, without guessing an author
condition assignment. Channel PSDs are computed with MNE from the same retained hu-neuro
epochs, then cached with the epoch file metadata and analysis settings.

In [ ]:
epoch_files = sorted((RESULTS / 'epochs').glob('*_epo.fif'))
assert len(epoch_files) == len(paper_subjects)
recipe = dict(method='welch', fmin=1, fmax=40, n_fft=256, n_overlap=128,
              picks='eeg', retained='P300 not missing', mne=mne.__version__,
              config_sha256=hashlib.sha256((RESULTS / 'config.json').read_bytes()).hexdigest(),
              files=[dict(name=p.name, bytes=p.stat().st_size, mtime_ns=p.stat().st_mtime_ns) for p in epoch_files])
signature = hashlib.sha256(json.dumps(recipe, sort_keys=True).encode()).hexdigest()
cache = PAPER_DIR / 'participant_channel_psd.npz'
cache_valid = False
if cache.exists():
    with np.load(cache, allow_pickle=False) as saved:
        cache_valid = str(saved['signature']) == signature
        if cache_valid:
            channel_psds = saved['power_v2_hz']
            psd_frequencies = saved['frequencies_hz']
            psd_channels = saved['channels'].tolist()
if not cache_valid:
    channel_psds = []
    for index, filename in enumerate(epoch_files, 1):
        print(f'Channel PSD {index}/{len(epoch_files)}: {filename.name[:7]}', flush=True)
        epochs = mne.read_epochs(filename, preload=True, verbose='ERROR').pick('eeg')
        assert len(epochs.ch_names) == 64
        per_condition = []
        for condition in PAPER_CONDITIONS:
            selected = np.flatnonzero(epochs.metadata.P300.notna() & epochs.metadata.condition.eq(condition))
            assert len(selected) > 0
            spectrum = epochs[selected].compute_psd(method='welch', fmin=1, fmax=40,
                n_fft=256, n_overlap=128, n_jobs=1, picks='eeg', verbose='ERROR')
            power, psd_frequencies = spectrum.get_data(return_freqs=True)
            per_condition.append(power.mean(axis=0))
        if index == 1:
            psd_channels = epochs.ch_names
        else:
            assert epochs.ch_names == psd_channels
        channel_psds.append(per_condition)
        del epochs, spectrum, power
    channel_psds = np.array(channel_psds)
    np.savez_compressed(cache, signature=np.array(signature), power_v2_hz=channel_psds,
                        frequencies_hz=psd_frequencies, channels=np.array(psd_channels))
    (PAPER_DIR / 'channel_psd_recipe.json').write_text(json.dumps(recipe, indent=2), encoding='utf-8')

# Check that the new spatial calculation recovers the existing mean PSD figures.
for ci, condition in enumerate(PAPER_CONDITIONS):
    existing = paper_psd.loc[paper_psd.condition.eq(condition)]
    np.testing.assert_allclose(existing.power_uv2_hz.to_numpy(),
                               channel_psds[:,ci].mean(axis=(0,1)) * 1e12, rtol=1e-8, atol=1e-10)
print('Spatial PSDs agree with the existing hu-neuro epoch-based power figures.')

In [ ]:
map_info = paper_grands['oddball'].copy().pick('eeg').info
assert map_info.ch_names == psd_channels
band = (psd_frequencies >= 4) & (psd_frequencies <= 30)
band_power = channel_psds[:,:,:,band].mean(axis=(0,3)) * 1e12
band_power_db = 10 * np.log10(band_power)
band_rows = []

def single_paper_map(values, title, unit, name, limits):
    fig = plt.figure(figsize=(7.08, 4.65))
    ax = fig.add_axes([.035,.025,.67,.90])
    color_ax = fig.add_axes([.89,.08,.025,.84])
    image, _ = mne.viz.plot_topomap(values, map_info, axes=ax, show=False, cmap='jet',
                                   contours=6, sensors='k.', extrapolate='head', vlim=limits)
    ax.set_title(title, fontsize=10, fontweight='bold')
    colorbar = fig.colorbar(image, cax=color_ax)
    colorbar.set_label(unit)
    colorbar.set_ticks(np.linspace(limits[0], limits[1], 6))
    return save_paper_figure(fig, name)

for ci, condition in enumerate(PAPER_CONDITIONS):
    values = band_power_db[ci]
    single_paper_map(values, f'Spectrum - 4–30 Hz ({condition})', 'dB re 1 µV²/Hz',
                     f'spectrum_4_30hz_{condition}', (float(band_power_db.min()), float(band_power_db.max())))
    band_rows.append(pd.DataFrame(dict(condition=condition, channel=psd_channels, power_db=values)))
    for number, seconds in [(4,.020), (5,.050)]:
        evoked = paper_grands[condition].copy().pick('eeg')
        index = np.abs(evoked.times-seconds).argmin()
        values = evoked.data[:,index] * 1e6
        limit = float(np.abs(values).max())
        single_paper_map(values, f'ERP - {seconds*1000:g} ms ({condition})', 'µV',
                         f'figure{number:02d}_erp_{condition}', (-limit,limit))
plt.show()
pd.concat(band_rows,ignore_index=True).to_csv(PAPER_DIR / 'spectrum_4_30hz_by_channel.csv',index=False)

for number, time_ms in [(4,20), (5,50)]:
    fig, axes = plt.subplots(2,3,figsize=(15,9))
    images = [REFERENCES/f'figure_{number:02d}.jpg',
              PAPER_DIR/'spectrum_4_30hz_standard.png', PAPER_DIR/'spectrum_4_30hz_oddball.png',
              None, PAPER_DIR/f'figure{number:02d}_erp_standard.png', PAPER_DIR/f'figure{number:02d}_erp_oddball.png']
    titles = ['Published reference', 'Spectrum interpretation: standard', 'Spectrum interpretation: oddball',
              '', f'Caption interpretation: standard, {time_ms} ms', f'Caption interpretation: oddball, {time_ms} ms']
    for ax, filename, title in zip(axes.flat, images, titles):
        if filename is not None:
            ax.imshow(plt.imread(filename))
            ax.set_title(title, fontsize=11)
        ax.axis('off')
    axes[1,0].text(.05,.6,'Image title and caption specify different statistics.\nBoth are shown; author condition mapping is unknown.\nNo voltage or power offset is added.', transform=axes[1,0].transAxes,fontsize=11,va='center')
    fig.text(.5,.01,'Original: Attar (2025), doi:10.3389/fnhum.2025.1616456, CC BY 4.0. Other panels: recomputed local data.',ha='center',fontsize=9)
    fig.tight_layout(rect=(0,.03,1,1))
    save_paper_figure(fig,f'comparison_figure{number:02d}')
    plt.show()

## Remaining published figures and reproduction status

Figure 1 is a task illustration. Figure 7 requires meditation/cognitive labels and classifier
predictions absent from these local files. Figures 8–11 are labeled simulations; their source
arrays/code are not supplied here. They remain original references, rather than invented EEG
results. The next cell displays them with attribution and saves a figure-by-figure status table.

Manual channel/time/ICA exclusions used by the authors are also unavailable. Our existing
ICA-off, 200 µV rejection settings therefore remain explicit differences. The original curve
estimates and side-by-side figures describe observed discrepancies; they do not establish
which unpublished processing choice caused them.

In [ ]:
status = pd.DataFrame([
    (1,'Reference only','Task illustration; original attributed asset.'),
    (2,'Layout match; mapping differs','Five local subjects shown; author P1–P5 mapping unknown. All-subject companion included.'),
    (3,'Recomputed plus approximate numerical comparison','Pz chosen explicitly; image-derived reference traces and maxima saved.'),
    (4,'Ambiguous statistic; both interpretations shown','Image: 4–30 Hz spectrum. Caption: 20 ms ERP. Author condition unknown.'),
    (5,'Ambiguous statistic; both interpretations shown','Image: 4–30 Hz spectrum. Caption: 50 ms ERP. Author condition unknown.'),
    (6,'Recomputed plus scale comparison','Reference power normalization unspecified; local PSD units are explicit.'),
    (7,'Reference only; labels missing','No local meditation/cognitive targets or author predictions.'),
    (8,'Reference only; simulation','No source simulation array/code available locally.'),
    (9,'Reference only; simulation','No source simulation array/code available locally.'),
    (10,'Reference only; simulation','No source simulation array/code available locally.'),
    (11,'Reference only; simulation','No source simulation array/code available locally.'),
],columns=['paper_figure','status','reason'])
status.to_csv(PAPER_DIR/'reproduction_status.csv',index=False)
display(status)
for number in [1,7,8,9,10,11]:
    fig, ax = plt.subplots(figsize=(8,5))
    ax.imshow(plt.imread(REFERENCES/f'figure_{number:02d}.jpg'))
    ax.set_title(f'Original published Figure {number} — reference only',fontsize=12)
    ax.axis('off')
    fig.text(.5,.015,'Attar (2025), doi:10.3389/fnhum.2025.1616456. CC BY 4.0. Not recomputed from local EEG.',ha='center',fontsize=9)
    fig.tight_layout(rect=(0,.035,1,1))
    plt.show()

print('Paper-style figures, comparisons and numerical tables:',PAPER_DIR)
print('Original reference assets and source/license manifest:',REFERENCES)